In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, accuracy_score,
    roc_curve, auc,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
    precision_recall_curve,
    average_precision_score
)

from imblearn.combine import SMOTETomek
from xgboost import XGBClassifier
import shap

####

class Message:
    def __init__(self, sender, receiver, content, performative="inform"):
        self.sender = sender
        self.receiver = receiver
        self.content = content
        self.performative = performative


class Agent:
    def __init__(self, name):
        self.name = name

    def send(self, message, receiver):
        receiver.receive(message)

    def receive(self, message):
        pass

######

class DataAgent(Agent):
    def __init__(self, path, target):
        super().__init__("DataAgent")
        self.path = path
        self.target = target
        self.encoders = {}
        self.selected_features = None  # Armazenará o nome das 10 melhores colunas

    def load_data(self):
        """Carrega os dados e remove índices desnecessários."""
        df = pd.read_csv(self.path)
        df = df.drop(columns=["Unnamed: 0"], errors="ignore")
        return df

    def preprocess(self, df):
        """Executa limpeza, encoding e imputação."""
        # Label Encoding para variáveis categóricas
        for col in df.select_dtypes(include="object").columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            self.encoders[col] = le

        # Imputação de valores ausentes pela mediana
        df = df.fillna(df.median(numeric_only=True))
        return df

    def select_features(self, X, y, top_k=10):
        """
        Rigor Metodológico: Seleção de Atributos via Feature Importance do XGBoost.
        Filtra apenas as variáveis com maior poder preditivo.
        """
        # Modelo temporário para extração de importância
        selector_model = XGBClassifier(random_state=42, eval_metric="logloss")
        selector_model.fit(X, y)

        # Identificação dos índices das top_k colunas
        importances = selector_model.feature_importances_
        indices = np.argsort(importances)[-top_k:]

        # Armazena as colunas selecionadas para uso posterior no split/teste
        self.selected_features = X.columns[indices].tolist()

        print(f"[{self.name}] Rigor aplicado: {top_k} variáveis selecionadas.")
        return X[self.selected_features]

    def split(self, df):
        """Divide os dados em treino e teste com estratificação."""
        X = df.drop(columns=[self.target])
        y = df[self.target]

        return train_test_split(
            X, y,
            test_size=0.3,
            random_state=42,
            stratify=y
        )
######

class PredictiveAgent(Agent):
    def __init__(self):
        super().__init__("PredictiveAgent")
        self.model = XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        )

    def train(self, X, y):
        self.model.fit(X, y)

    def predict_proba(self, X):
        return self.model.predict_proba(X)[:, 1]
#####

class EvaluationAgent(Agent):
    def __init__(self):
        super().__init__("EvaluationAgent")

    def find_best_threshold(self, y_true, y_prob):
        thresholds = np.linspace(0.05, 0.95, 91)
        best_t, best_f1 = 0.5, 0

        for t in thresholds:
            y_pred = (y_prob >= t).astype(int)
            f1 = f1_score(y_true, y_pred, average="macro")

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        return best_t, best_f1
#######

class ExplainabilityAgent(Agent):
    def __init__(self, model):
        super().__init__("ExplainabilityAgent")
        self.explainer = shap.TreeExplainer(model)

    def global_importance(self, X):
        shap_values = self.explainer.shap_values(X)
        shap_used = shap_values[1] if isinstance(shap_values, list) else shap_values
        importance = np.abs(shap_used).mean(axis=0)

        return pd.DataFrame({
            "Feature": X.columns,
            "Importance": importance
        }).sort_values(by="Importance", ascending=False)

    def local_explanation(self, X_instance):
        shap_values = self.explainer.shap_values(X_instance)
        shap_used = shap_values[1] if isinstance(shap_values, list) else shap_values

        return pd.Series(
            shap_used[0],
            index=X_instance.columns
        ).sort_values(key=abs, ascending=False)


#######

class DecisionSupportAgent(Agent):
    def __init__(self):
        super().__init__("DecisionSupportAgent")

    def decide(self, prob, threshold, explanation, top_k=5):
        decision = (
            "DESFECHO DESFAVORÁVEL"
            if prob >= threshold
            else "DESFECHO FAVORÁVEL"
        )

        return {
            "decisao": decision,
            "probabilidade": prob,
            "threshold": threshold,
            "principais_fatores": explanation.head(top_k).to_dict()
        }
######

class TuberculosisDecisionSystem:
    def __init__(self, data_path, target):
        self.data_agent = DataAgent(data_path, target)
        self.model_agent = PredictiveAgent()
        self.eval_agent = EvaluationAgent()
        self.decision_agent = DecisionSupportAgent()

    def run(self):
        """
        Executa o pipeline multiagente com rigor metodológico elevado,
        selecionando as 10 melhores características antes da modelagem.
        """
        # 1. Percepção e Carregamento (DataAgent)
        df = self.data_agent.load_data()

        # 2. Pré-processamento (DataAgent - Encoding e Imputação)
        df = self.data_agent.preprocess(df)

        # 3. Divisão dos Dados (DataAgent - Estratificado)
        X_train, X_test, y_train, y_test = self.data_agent.split(df)

        # 4. Rigor Metodológico: Seleção de Atributos (DataAgent - Embedded)
        # Seleciona as 10 melhores características baseadas no ganho de informação
        X_train_selected = self.data_agent.select_features(X_train, y_train, top_k=10)

        # Garante que o conjunto de teste tenha exatamente as mesmas colunas selecionadas
        X_test_selected = X_test[self.data_agent.selected_features]

        # 5. Balanceamento (SMOTETomek - Apenas no Treino Selecionado)
        sm = SMOTETomek(random_state=42)
        X_train_bal, y_train_bal = sm.fit_resample(X_train_selected, y_train)

        # 6. Aprendizado (PredictiveAgent)
        self.model_agent.train(X_train_bal, y_train_bal)

        # 7. Inferência (PredictiveAgent - Probabilidades)
        y_prob = self.model_agent.predict_proba(X_test_selected)

        # 8. Avaliação (EvaluationAgent - Busca do threshold ótimo para F1-Macro)
        best_t, best_f1 = self.eval_agent.find_best_threshold(y_test, y_prob)

        # 9. Explicabilidade (ExplainabilityAgent - SHAP com os dados reduzidos)
        expl_agent = ExplainabilityAgent(self.model_agent.model)
        global_xai = expl_agent.global_importance(X_test_selected)
        local_xai = expl_agent.local_explanation(X_test_selected.iloc[[0]])

        # 10. Apoio à Decisão (DecisionSupportAgent - Integração final)
        decision = self.decision_agent.decide(
            prob=y_prob[0],
            threshold=best_t,
            explanation=local_xai
        )

        return {
            "decision": decision,
            "X_test": X_test_selected, # Retorna os dados com as colunas corretas
            "y_test": y_test,
            "y_prob": y_prob,
            "best_threshold": best_t,
            "global_xai": global_xai,
            "model": self.model_agent.model,
            "selected_features": self.data_agent.selected_features # Lista das 10 vars
        }
#######

system = TuberculosisDecisionSystem(
    data_path="/content/tuberculosis.csv",
    target="sitAtual"
)

results = system.run()
decision_report = results["decision"]
X_test = results["X_test"]
y_test = results["y_test"]
y_prob = results["y_prob"]
model = results["model"]
best_threshold = results["best_threshold"]

decision_report


#######

y_pred = (y_prob >= best_threshold).astype(int)

print("F1-score (macro):", f1_score(y_test, y_pred, average="macro"))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nRelatório de Classificação:\n")
print(classification_report(y_test, y_pred))


from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

# ======================
# Curva ROC (modelo avaliado)
# ======================
#fpr, tpr, _ = roc_curve(y_test, y_prob)
#roc_auc = auc(fpr, tpr)

####################

import matplotlib.pyplot as plt
import seaborn as sns

# Seleciona as 10 principais variáveis
top_features = results["global_xai"].head(10)

# Define uma paleta de cores diferenciadas
colors = sns.color_palette("Set2", len(top_features))

plt.figure(figsize=(7,5))
plt.barh(
    top_features["Feature"],
    top_features["Importance"],
    color=colors
)
plt.gca().invert_yaxis()
plt.xlabel("Impacto médio na decisão")
#plt.title("Principais Variáveis Explicativas (XAI Global)")
plt.tight_layout()
# Salvar uma única figura com as duas matrizes
plt.savefig("Principais Variáveis Explicativas (XAI Global).png", dpi=300, bbox_inches='tight')
plt.show()

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Criação do DataFrame a partir dos fatores
fatores = decision_report["principais_fatores"]
fatores_df = pd.DataFrame.from_dict(
    fatores, orient="index", columns=["Contribuição SHAP"]
).sort_values(by="Contribuição SHAP")

# Paleta de cores consistente (Set2)
colors = sns.color_palette("Set2", len(fatores_df))

plt.figure(figsize=(7,5))
plt.barh(
    fatores_df.index,
    fatores_df["Contribuição SHAP"],
    color=colors
)
plt.axvline(0, linestyle="--", color="black", linewidth=1)
plt.xlabel("Contribuição para a decisão")
#plt.title("Explicação da Decisão Individual")
plt.tight_layout()
plt.savefig("Explicação da Decisão Individual.png", dpi=300, bbox_inches='tight')
plt.show()

import matplotlib.pyplot as plt

# Probabilidade estimada
prob = decision_report["probabilidade"]

plt.figure(figsize=(4,4))
plt.bar(["Probabilidade estimada"], [prob], color="skyblue")  # <- cor definida
plt.ylim(0,1)
#plt.title("Probabilidade Estimada de Desfecho Desfavorável")
plt.text(0, prob - 0.1, f"{prob:.2%}", ha="center", fontsize=12)
plt.tight_layout()
plt.savefig("Probabilidade Estimada de Desfecho Desfavorável.png", dpi=300, bbox_inches='tight')
plt.show()

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

plt.figure(figsize=(5,5))
disp.plot(cmap="Blues", values_format="d")
plt.title("Matriz de Confusão – Sistema Multiagente")
plt.tight_layout()
plt.show()



##### Comparação A e B #########


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from imblearn.combine import SMOTETomek

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)

import shap

##########

df = system.data_agent.load_data()
df = system.data_agent.preprocess(df)

X_train, X_test, y_train, y_test = system.data_agent.split(df)


#########

############

model_no_balance = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

model_no_balance.fit(X_train, y_train)

y_prob_nb = model_no_balance.predict_proba(X_test)[:, 1]
y_pred_nb = (y_prob_nb >= 0.5).astype(int)

print("SEM balanceamento\n")
print(classification_report(y_test, y_pred_nb))

f1_macro_nb = f1_score(y_test, y_pred_nb, average="macro")
acc_nb = accuracy_score(y_test, y_pred_nb)

#############

smote_tomek = SMOTETomek(random_state=42)
X_train_bal, y_train_bal = smote_tomek.fit_resample(X_train, y_train)

model_smote = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

model_smote.fit(X_train_bal, y_train_bal)

y_prob_sm = model_smote.predict_proba(X_test)[:, 1]
y_pred_sm = (y_prob_sm >= 0.5).astype(int)

print("COM Balanceamento\n")
print(classification_report(y_test, y_pred_sm))

f1_macro_sm = f1_score(y_test, y_pred_sm, average="macro")
acc_sm = accuracy_score(y_test, y_pred_sm)

###########

results_df = pd.DataFrame({
    "Modelo": ["Sem balanceamento", "Com Balanceamento"],
    "F1-score (macro)": [f1_macro_nb, f1_macro_sm],
    "Accuracy": [acc_nb, acc_sm]
})

results_df

############


def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 91)
    best_t, best_f1 = 0.5, 0

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, average="macro")

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1
################


best_t, best_f1 = find_best_threshold(y_test, y_prob_sm)

print(f"Melhor threshold (Com balanceamento): {best_t:.2f}")
print(f"F1 macro otimizado: {best_f1:.3f}")

y_pred_sm_opt = (y_prob_sm >= best_t).astype(int)
print("\nRelatório com threshold otimizado:\n")
print(classification_report(y_test, y_pred_sm_opt))

############


precision, recall, _ = precision_recall_curve(y_test, y_prob_sm)
ap = average_precision_score(y_test, y_prob_sm)

plt.figure(figsize=(6,5))
plt.plot(recall, precision, label=f'AP = {ap:.3f}')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision–Recall (Classe Positiva)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


###########

explainer_nb = shap.TreeExplainer(model_no_balance)
shap_values_nb = explainer_nb.shap_values(X_test)

if isinstance(shap_values_nb, list):
    shap_nb = shap_values_nb[1]
else:
    shap_nb = shap_values_nb

importance_nb = np.abs(shap_nb).mean(axis=0)

##############
explainer_sm = shap.TreeExplainer(model_smote)
shap_values_sm = explainer_sm.shap_values(X_test)

if isinstance(shap_values_sm, list):
    shap_sm = shap_values_sm[1]
else:
    shap_sm = shap_values_sm

importance_sm = np.abs(shap_sm).mean(axis=0)

#############


shap_compare = pd.DataFrame({
    "Feature": X_test.columns,
    "Sem balanceamento": importance_nb,
    "Com balanceamento": importance_sm
}).set_index("Feature")

shap_compare["Importância"] = (
    shap_compare["Com balanceamento"] - shap_compare["Sem balanceamento"]
)

shap_compare.sort_values("Importância", ascending=False).head(10)

#############

shap.summary_plot(shap_sm, X_test, show=True)


##################

################

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Criar subplots lado a lado
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Primeira matriz de confusão (sem balanceamento)
cm_nb = confusion_matrix(y_test, y_pred_nb)
disp_nb = ConfusionMatrixDisplay(cm_nb)
disp_nb.plot(ax=axes[0], cmap="viridis", values_format="d")  # cores iguais às da imagem
axes[0].set_title("Matriz de Confusão - Sem balanceamento")

# Segunda matriz de confusão (SMOTETomek)
cm_sm = confusion_matrix(y_test, y_pred_sm)
disp_sm = ConfusionMatrixDisplay(cm_sm)
disp_sm.plot(ax=axes[1], cmap="viridis", values_format="d")  # cores iguais às da imagem
axes[1].set_title("Matriz de Confusão - Com balanceamento")

# Ajustar layout
plt.tight_layout()

# Salvar uma única figura com as duas matrizes
plt.savefig("confusion_matrix_comparacao.png", dpi=300, bbox_inches='tight')

# Mostrar
plt.show()
plt.close()


########################


from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

# ======================
# ROC
# ======================
fpr_nb, tpr_nb, _ = roc_curve(y_test, y_prob_nb)
auc_nb = auc(fpr_nb, tpr_nb)

fpr_sm, tpr_sm, _ = roc_curve(y_test, y_prob_sm)
auc_sm = auc(fpr_sm, tpr_sm)

# ======================
# Precision-Recall
# ======================
precision_nb, recall_nb, _ = precision_recall_curve(y_test, y_prob_nb)
ap_nb = average_precision_score(y_test, y_prob_nb)

precision_sm, recall_sm, _ = precision_recall_curve(y_test, y_prob_sm)
ap_sm = average_precision_score(y_test, y_prob_sm)

# ======================
# Figura comparativa final
# ======================
plt.figure(figsize=(12,5))

# --- Subplot 1: ROC ---
plt.subplot(1, 2, 1)
plt.plot(fpr_nb, tpr_nb, label=f'Sem balanceamento (AUC={auc_nb:.3f})')
plt.plot(fpr_sm, tpr_sm, label=f'Com balanceamento (AUC={auc_sm:.3f})')
plt.plot([0,1], [0,1], linestyle='--', color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Comparação ROC")
plt.legend()
plt.grid(True)

# --- Subplot 2: Precision–Recall ---
plt.subplot(1, 2, 2)
plt.plot(recall_nb, precision_nb, label=f'Sem balanceamento (AP={ap_nb:.3f})')
plt.plot(recall_sm, precision_sm, label=f'Com balanceamento (AP={ap_sm:.3f})')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Comparação Precision–Recall")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("Curva_ROC_PR_Comparacao.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close()



